# 03 — DistilBERT Error Analysis

This notebook explains where the corrected-UTC DistilBERT model disagrees with rating-derived weak labels. It reads the new predictions from Notebook `02`; it does not train or run either model again.

## Objectives

- Measure errors separately for products excluded from training, later reviews, and the first `Conditioners` niche.
- Check the Neutral class, review length, shortened long reviews, products, dates, and Amazon categories.
- Measure whether a large saved probability really corresponds to a high share of correct answers.
- Find possible disagreements between review text and star rating, mixed opinions, and delivery or seller comments.
- Compare DistilBERT and TF-IDF on the same reviews.
- Create a new versioned review queue without copying historical human labels.

The star-based labels are weak labels: 1–2 stars mean Negative, 3 means Neutral, and 4–5 mean Positive. An error against this label is exact, but it does not always mean that the model misunderstood the text. The buyer may select the wrong rating, express several opinions, discuss delivery, or write something ambiguous.

Seller attention is kept separate from text sentiment: 1–3 stars mean `needs_attention`, while 4–5 stars mean `satisfied`. This is a transparent business rule, not another ML model.

# Анализ ошибок DistilBERT

Этот ноутбук показывает, где исправленная UTC-версия DistilBERT расходится со слабыми метками по рейтингу. Он читает новые результаты ноутбука `02`, поэтому повторно обучать или запускать модели не нужно.

## Цели

- Отдельно измерить ошибки для товаров, отзывы о которых не использовались при обучении, более поздних отзывов и первой ниши `Conditioners`.
- Проверить класс Neutral, длину отзывов, сокращённые длинные отзывы, товары, даты и категории Amazon.
- Измерить, действительно ли большая сохранённая вероятность соответствует большой доле правильных ответов.
- Найти возможные несовпадения текста и количества звёзд, смешанные мнения и замечания о доставке или продавце.
- Сравнить DistilBERT и TF-IDF на одних и тех же отзывах.
- Создать новую версионированную очередь для человека без переноса исторических меток.

Метки созданы по звёздам: 1–2 звезды означают Negative, 3 — Neutral, 4–5 — Positive. Несовпадение ответа модели с такой меткой считается точно, но оно не всегда означает, что модель неправильно поняла текст. Покупатель мог ошибиться с оценкой, высказать несколько мнений, обсуждать доставку или написать неоднозначный текст.

Сигнал продавцу хранится отдельно от тональности текста: 1–3 звезды означают `needs_attention`, а 4–5 звёзд — `satisfied`. Это прозрачное бизнес-правило, а не ещё одна ML-модель.

In [ ]:
# Standard library / Стандартная библиотека
import json
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

# Third-party packages / Сторонние библиотеки
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Locate the repository before importing reusable project code.
# Находим репозиторий до импорта переиспользуемого кода проекта.
PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "PLAN.md").is_file() and (candidate / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Local project modules / Локальные модули проекта
from src.common.csv_safety import escape_dataframe_for_spreadsheet
from src.ml.error_analysis import (
    MANUAL_LABEL_COLUMNS,
    add_error_features,
    build_manual_review_queue,
    calibration_report,
    class_metric_table,
    compare_saved_models,
    error_rates_by_group,
    json_ready,
    load_error_analysis_rows,
    preserve_manual_review_labels,
    summarize_manual_review,
)
from src.ml.sentiment import SENTIMENT_LABELS

## 1. Fixed inputs and outputs

The analysis uses three final test groups. The validation rows used while choosing the training checkpoint are intentionally excluded from the final error counts.

The detailed error table is saved as Parquet inside processed data. A compact JSON report is saved without review text. A new Parquet/CSV queue is created for this model version; until a person completes it, the JSON summary records `pending_human_review` and makes no human-agreement claim.

## Фиксированные входные и выходные файлы

Анализ использует три итоговые проверочные группы. Строки Validation, которые использовались при выборе сохранённой точки обучения, намеренно не входят в итоговый подсчёт ошибок.

Подробная таблица ошибок сохраняется в формате Parquet среди обработанных данных. Краткий JSON-отчёт не содержит тексты отзывов. Для этой версии модели создаётся новая Parquet/CSV-очередь; пока человек её не завершил, JSON-summary имеет статус `pending_human_review` и не содержит claims о согласии с человеком.

In [ ]:
# Keep every version in the paths so later model runs cannot overwrite this analysis.
# Указываем версии прямо в путях, чтобы будущие модели не перезаписали этот анализ.
DATASET_VERSION = "amazon_reviews_2023_beauty_2021_2023_v1"
SPLIT_VERSION = "beauty_rating_sentiment_split_1m_v3"
MODEL_VERSION = "beauty_distilbert_rating_sentiment_1m_v2"
BASELINE_VERSION = "beauty_tfidf_rating_sentiment_1m_v3"
RANDOM_STATE = 42
EVALUATION_SPLITS = (
    "test_product",
    "test_temporal_seen",
    "test_conditioners",
)
SPLIT_NAMES = {
    "test_product": "Products excluded from training",
    "test_temporal_seen": "Later 2023 reviews",
    "test_conditioners": "Held-out Conditioner parent products",
}

DATASET_DIRECTORY = PROJECT_ROOT / "data/processed" / DATASET_VERSION
REVIEWS_PATH = DATASET_DIRECTORY / "reviews.parquet"
CATALOG_PATH = DATASET_DIRECTORY / "product_catalog.parquet"
PREDICTIONS_PATH = (
    DATASET_DIRECTORY / "model_predictions" / f"{MODEL_VERSION}.parquet"
)
BASELINE_PREDICTIONS_PATH = (
    DATASET_DIRECTORY / "model_predictions" / f"{BASELINE_VERSION}.parquet"
)
MODEL_EVALUATION_PATH = (
    PROJECT_ROOT / "reports/model_evaluation" / f"{MODEL_VERSION}.json"
)
BASELINE_EVALUATION_PATH = (
    PROJECT_ROOT / "reports/model_evaluation" / f"{BASELINE_VERSION}.json"
)
ERROR_DIRECTORY = DATASET_DIRECTORY / "error_analysis"
ERROR_ROWS_PATH = ERROR_DIRECTORY / f"{MODEL_VERSION}_errors.parquet"
MANUAL_REVIEW_PATH = (
    ERROR_DIRECTORY / f"{MODEL_VERSION}_manual_review_queue.parquet"
)
MANUAL_REVIEW_CSV_PATH = (
    PROJECT_ROOT
    / "notebooks/02_sentiment"
    / f"{MODEL_VERSION}_manual_review_queue.csv"
)
MANUAL_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports/model_evaluation"
    / f"{MODEL_VERSION}_manual_review_summary.json"
)
REPORT_PATH = (
    PROJECT_ROOT
    / "reports/model_evaluation"
    / f"{MODEL_VERSION}_error_analysis.json"
)

display(
    pd.Series(
        {
            "python": platform.python_version(),
            "dataset_version": DATASET_VERSION,
            "split_version": SPLIT_VERSION,
            "model_version": MODEL_VERSION,
            "baseline_version": BASELINE_VERSION,
            "random_state": RANDOM_STATE,
            "evaluation_splits": ", ".join(EVALUATION_SPLITS),
        },
        name="value",
    ).to_frame()
)

## 2. Load saved answers and restore review text

Each prediction is joined to the canonical review by `review_id` and to the Amazon product catalog by `parent_asin`. The join must retain every selected prediction; otherwise the notebook stops instead of silently analyzing incomplete data.

## Загружаем сохранённые ответы и тексты

Каждое предсказание соединяется с исходным очищенным отзывом по `review_id` и с каталогом товаров Amazon по `parent_asin`. После соединения должны сохраниться все выбранные предсказания. Если часть данных потеряется, ноутбук остановится, а не продолжит анализ неполной таблицы.

In [ ]:
# Validate model lineage before reading final-test rows.
# Проверяем lineage моделей до чтения итоговых test-строк.
model_evaluation = json.loads(
    MODEL_EVALUATION_PATH.read_text(encoding="utf-8")
)
baseline_evaluation = json.loads(
    BASELINE_EVALUATION_PATH.read_text(encoding="utf-8")
)
assert model_evaluation["dataset_version"] == DATASET_VERSION
assert model_evaluation["split_version"] == SPLIT_VERSION
assert model_evaluation["model_config"]["model_version"] == MODEL_VERSION
assert baseline_evaluation["dataset_version"] == DATASET_VERSION
assert baseline_evaluation["split_config"]["split_version"] == SPLIT_VERSION
assert baseline_evaluation["model_config"]["model_version"] == (
    BASELINE_VERSION
)
analysis_rows = load_error_analysis_rows(
    PREDICTIONS_PATH,
    REVIEWS_PATH,
    CATALOG_PATH,
    split_names=EVALUATION_SPLITS,
)
analysis_rows = add_error_features(analysis_rows)

assert len(analysis_rows) == 209_021
assert analysis_rows["review_id"].is_unique
assert set(analysis_rows["sentiment_label"]) == set(SENTIMENT_LABELS)
assert analysis_rows["model_score"].between(0, 1).all()
assert analysis_rows["seller_attention"].isin(
    ["needs_attention", "satisfied"]
).all()

split_counts = (
    analysis_rows["split_name"]
    .value_counts()
    .rename_axis("split_name")
    .rename("review_count")
    .reset_index()
)
split_counts["meaning"] = split_counts["split_name"].map(SPLIT_NAMES)
seller_attention_summary = (
    analysis_rows["seller_attention"]
    .value_counts()
    .rename_axis("seller_attention")
    .rename("review_count")
    .reset_index()
)
seller_attention_summary["review_share"] = (
    seller_attention_summary["review_count"] / len(analysis_rows)
)
display(split_counts)
display(seller_attention_summary)

## 3. How often and where does the model fail?

Accuracy answers a simple question: what share of answers exactly matches the label created from the star rating? Error rate is the remaining share. We show results by test group and true label so that the large Positive class cannot hide weaker results.

Three class measurements answer different questions. Precision asks: when the model gives this answer, how often does it match the stars? Recall asks: what share of reviews with this star-derived label did the model find? F1 combines both questions into one number. This difference is especially important for Neutral.

## Как часто и где ошибается модель?

Accuracy показывает долю ответов, которые точно совпали с меткой, созданной по звёздам. Доля ошибок показывает оставшуюся часть. Результаты разделены по проверочным группам и исходным меткам, чтобы большой класс Positive не скрывал более слабые результаты.

Три показателя класса отвечают на разные вопросы. Precision: когда модель даёт такой ответ, как часто он совпадает со звёздами? Recall: какую долю отзывов с такой меткой по звёздам модель нашла? F1 объединяет оба вопроса в одно число. Для Neutral эта разница особенно важна.

In [ ]:
# Separate tables answer whether errors come from a test group or one sentiment class.
# Отдельные таблицы показывают, связаны ли ошибки с проверочной группой или классом.
split_error_rates = error_rates_by_group(analysis_rows, "split_name")
split_error_rates["meaning"] = split_error_rates["split_name"].map(SPLIT_NAMES)
class_error_rates = error_rates_by_group(
    analysis_rows, ["split_name", "sentiment_label"]
)
class_metrics = class_metric_table(analysis_rows)

display(
    split_error_rates[
        [
            "split_name",
            "meaning",
            "review_count",
            "error_count",
            "accuracy",
            "error_rate",
        ]
    ]
)
display(class_error_rates)
display(class_metrics)

In [ ]:
# A row-normalized matrix shows where each rating-derived class is redirected.
# Матрица по долям показывает, с каким ответом модель путает каждую исходную метку.
confusion_share = pd.crosstab(
    analysis_rows["sentiment_label"],
    analysis_rows["predicted_label"],
    normalize="index",
).reindex(index=SENTIMENT_LABELS, columns=SENTIMENT_LABELS, fill_value=0)

plt.figure(figsize=(8, 5))
sns.heatmap(confusion_share, annot=True, fmt=".1%", cmap="Blues", vmin=0, vmax=1)
plt.title("DistilBERT answers by rating-derived label")
plt.xlabel("Model answer")
plt.ylabel("Label created from star rating")
plt.tight_layout()
plt.show()

error_pairs = (
    analysis_rows.loc[analysis_rows["is_error"], "error_pair"]
    .value_counts()
    .rename_axis("error_pair")
    .rename("error_count")
    .reset_index()
)
display(error_pairs)

## 4. Can the saved probability be trusted?

The model stores three probabilities and `model_score` is the largest of them. Before this check, we should not call that value confidence. For example, among answers with a score near 0.90, approximately 90% should be correct if the scores match reality well.

The expected calibration error is the weighted average difference between the model score and the observed correct-answer share; closer to zero is better. The Brier score and log loss also reward accurate probabilities and strongly penalize confidently wrong answers. We measure every answer type separately because a good overall average can hide a weak Neutral result.

## Можно ли доверять сохранённой вероятности?

Модель сохраняет три вероятности, а `model_score` — наибольшая из них. До этой проверки не следует называть это число уверенностью. Например, если оценки хорошо соответствуют реальности, среди ответов со значением около 0,90 примерно 90% должны быть правильными.

Ожидаемая ошибка соответствия вероятностей (expected calibration error) — средняя разница между оценкой модели и фактической долей правильных ответов; чем ближе к нулю, тем лучше. Brier score и log loss также оценивают все вероятности и сильнее наказывают модель за ошибочные ответы с очень большим значением. Каждый вид ответа проверяется отдельно, потому что хорошее среднее значение может скрыть слабый результат Neutral.

In [ ]:
# Measure all independent test rows together, then retain separate test-group checks.
# Сначала измеряем все независимые проверки вместе, затем сохраняем отдельные результаты.
calibration_table, calibration_summary = calibration_report(analysis_rows)
calibration_by_split = {}
for split_name, split_rows in analysis_rows.groupby("split_name", sort=True):
    _, split_summary = calibration_report(split_rows)
    calibration_by_split[split_name] = split_summary
calibration_by_answer = {}
for predicted_label, label_rows in analysis_rows.groupby(
    "predicted_label", sort=True
):
    _, label_summary = calibration_report(label_rows)
    calibration_by_answer[predicted_label] = label_summary

display(pd.Series(calibration_summary, name="value").to_frame())
display(calibration_table)
display(pd.DataFrame(calibration_by_answer).T)

plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], linestyle="--", color="black", label="Ideal match")
plt.plot(
    calibration_table["mean_model_score"],
    calibration_table["observed_accuracy"],
    marker="o",
    linewidth=2,
    label="DistilBERT",
)
plt.title("Saved probability versus observed accuracy")
plt.xlabel("Mean saved model probability")
plt.ylabel("Observed share of correct answers")
plt.xlim(0.3, 1.0)
plt.ylim(0.3, 1.0)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Review length and shortened long reviews

DistilBERT received at most 256 text parts called tokens. `was_truncated=True` means that the end of a longer review was not shown to the model. The comparison below uses the real token count saved during inference, not an approximate character count.

This comparison reveals an association, not a controlled experiment: long and short reviews can differ in many ways besides truncation.

## Длина и сокращение длинных отзывов

DistilBERT получала не более 256 частей текста, которые называются токенами. `was_truncated=True` означает, что конец длинного отзыва модель не увидела. Ниже используется настоящее количество токенов, сохранённое при запуске модели, а не приблизительное количество символов.

Это сравнение показывает связь, но не доказывает причину: длинные и короткие отзывы могут различаться не только из-за сокращения.

In [ ]:
# Keep shortened reviews separate because missing endings may contain the final opinion.
# Рассматриваем сокращённые отзывы отдельно: в пропущенном конце мог быть итоговый вывод.
length_error_rates = error_rates_by_group(analysis_rows, "token_length_group")
truncation_error_rates = error_rates_by_group(analysis_rows, "was_truncated")
display(length_error_rates)
display(truncation_error_rates)

plt.figure(figsize=(9, 5))
sns.barplot(
    data=length_error_rates,
    x="token_length_group",
    y="error_rate",
    color="#4C78A8",
)
plt.title("Error rate by review length seen during inference")
plt.xlabel("Token count before the 256-token limit")
plt.ylabel("Error rate")
plt.tight_layout()
plt.show()

## 6. Possible text–rating disagreements and difficult writing

Simple phrase lists flag suspicious cases such as a clearly positive phrase with 1–2 stars or a clearly negative phrase with 4–5 stars. Other indicators find contrast words, delivery or seller mentions, very short text, and strong punctuation or capital letters.

These are search aids, not new true labels. A phrase list misses implicit meaning, negation, sarcasm, and context. Counts are therefore called possible cases and must not be presented as proven buyer mistakes.

## Возможные несовпадения текста и звёзд

Простые списки выражений отмечают подозрительные случаи: например, явно положительную фразу при 1–2 звёздах или явно отрицательную при 4–5 звёздах. Другие признаки находят противопоставления, упоминания доставки или продавца, очень короткий текст, много восклицательных знаков или слова заглавными буквами.

Это подсказки для поиска, а не новые правильные метки. Список фраз не всегда понимает скрытый смысл, отрицание, сарказм и контекст. Поэтому ниже считаются возможные случаи, а не доказанные ошибки покупателей.

In [ ]:
# Report indicators among model errors and among all evaluated reviews for context.
# Показываем признаки среди ошибок модели и среди всех отзывов для сравнения.
INDICATOR_COLUMNS = [
    "possible_text_rating_disagreement",
    "contains_mixed_opinion_clue",
    "mentions_delivery_or_seller",
    "no_clear_dictionary_sentiment",
    "very_short_review",
    "strong_emotion_clue",
    "was_truncated",
]
model_errors = analysis_rows.loc[analysis_rows["is_error"]].copy()
indicator_summary = pd.DataFrame(
    {
        "possible_case_count_among_errors": model_errors[INDICATOR_COLUMNS].sum(),
        "share_among_errors": model_errors[INDICATOR_COLUMNS].mean(),
        "share_among_all_reviews": analysis_rows[INDICATOR_COLUMNS].mean(),
    }
).rename_axis("indicator").reset_index()
display(indicator_summary)

suspected_disagreements = model_errors.loc[
    model_errors["possible_text_rating_disagreement"],
    [
        "rating",
        "sentiment_label",
        "predicted_label",
        "model_score",
        "review_text",
    ],
].sort_values("model_score", ascending=False)
display(suspected_disagreements.head(12))

## 7. Products, dates, and Amazon categories

A product table is useful only when it has enough checked reviews. We require at least 20 test reviews before ranking a product by error rate. A high product error rate is a signal to inspect its reviews; it is not proof that the product itself is unusual.

Monthly and category summaries help detect a concentrated problem. Small groups are shown with their review counts so they are not mistaken for equally reliable estimates.

## Товары, даты и категории Amazon

Таблица по товарам полезна только при достаточном количестве проверочных отзывов. Для сравнения требуется не менее 20 отзывов на товар. Большая доля ошибок у товара означает, что его отзывы стоит проверить; это не доказывает, что сам товар необычный.

Сводки по месяцам и категориям помогают найти место, где сосредоточена проблема. Для небольших групп обязательно показано количество отзывов, чтобы их результаты не воспринимались как столь же надёжные.

In [ ]:
# Require enough examples before treating a product-level error rate as informative.
# Требуем достаточно примеров, прежде чем считать долю ошибок по товару полезной.
product_error_rates = error_rates_by_group(
    analysis_rows,
    ["parent_asin", "product_title", "category_path_text"],
)
ranked_products = (
    product_error_rates.loc[product_error_rates["review_count"].ge(20)]
    .sort_values(["error_rate", "review_count"], ascending=[False, False])
    .reset_index(drop=True)
)

analysis_rows["review_month"] = analysis_rows["review_timestamp"].dt.strftime("%Y-%m")
monthly_error_rates = error_rates_by_group(analysis_rows, "review_month")
category_error_rates = error_rates_by_group(
    analysis_rows, ["category_path_text", "leaf_category"]
).sort_values("review_count", ascending=False)

display(ranked_products.head(15))
display(monthly_error_rates)
display(category_error_rates.head(15))

## 8. Does DistilBERT improve on TF-IDF with the same data?

Both models were trained on the same 999,999 review IDs. Their saved predictions also contain the same validation and final-test review IDs. Joining by `review_id` verifies this equality and separates cases where both models are right, only one is right, or both are wrong.

Cases where both text models disagree with the stars are especially useful for manual review because they can contain an inaccurate star-derived label or genuinely difficult wording.

## Улучшает ли DistilBERT результат TF-IDF на тех же данных?

Обе модели обучались на одинаковых 999 999 `review_id`. Их сохранённые ответы также содержат одинаковые строки для настройки и итоговой проверки. Соединение по `review_id` подтверждает равенство и отдельно считает случаи, когда обе модели правы, права только одна модель или ошиблись обе.

Случаи, где обе текстовые модели не совпали со звёздами, особенно полезны для ручной проверки: там может быть неточная метка по звёздам или действительно сложная формулировка.

In [ ]:
# Load the small saved prediction table, not the fitted TF-IDF model.
# Загружаем небольшую таблицу ответов, а не обученную модель TF-IDF.
baseline_predictions = pd.read_parquet(
    BASELINE_PREDICTIONS_PATH,
    filters=[("split_name", "in", EVALUATION_SPLITS)],
)
model_comparison_rows, model_comparison_summary = compare_saved_models(
    analysis_rows,
    baseline_predictions,
)
comparison_coverage = (
    analysis_rows.groupby("split_name").size().rename("DistilBERT rows").to_frame()
    .join(
        model_comparison_rows.groupby("split_name")
        .size()
        .rename("rows present in both files")
    )
    .reset_index()
)
comparison_coverage["shared_row_share"] = (
    comparison_coverage["rows present in both files"]
    / comparison_coverage["DistilBERT rows"]
)
assert comparison_coverage["shared_row_share"].eq(1.0).all()
comparison_pivot = (
    model_comparison_summary.pivot(
        index="split_name",
        columns="comparison_result",
        values="review_count",
    )
    .fillna(0)
    .astype(int)
)
display(comparison_coverage)
display(comparison_pivot)

## 9. Prepare examples for review by a person

The worksheet takes up to 25 errors from each suggested group with a fixed random seed. It is a fresh queue tied to this exact model, baseline, and split version. Labels from the historical queue are not copied because changed predictions and membership create a different diagnostic sample.

Recommended values are `negative`, `neutral`, `positive`, `mixed`, or `unclear` for the text; `yes`, `no`, or `unclear` for rating agreement; and a short cause such as `model error`, `rating error`, `mixed opinion`, `delivery/seller`, `sarcasm`, `truncation`, or `unclear`.

## Готовим примеры для проверки человеком

В таблицу попадает до 25 ошибок из каждой предполагаемой группы с фиксированным seed. Это новая очередь, связанная с точными версиями модели, baseline и split. Метки исторической очереди не копируются: изменившиеся predictions и membership создают другую диагностическую выборку.

Для смысла текста рекомендуется использовать `negative`, `neutral`, `positive`, `mixed` или `unclear`; для соответствия звёзд — `yes`, `no` или `unclear`; для причины — `model error`, `rating error`, `mixed opinion`, `delivery/seller`, `sarcasm`, `truncation` или `unclear`.

In [ ]:
# Rebuild deterministically; preserve labels only from this exact new lineage.
# Пересобираем детерминированно; сохраняем метки только этого нового lineage.
manual_review_queue = build_manual_review_queue(
    analysis_rows,
    rows_per_group=25,
    random_state=RANDOM_STATE,
)
manual_review_queue.insert(0, "dataset_version", DATASET_VERSION)
manual_review_queue.insert(1, "split_version", SPLIT_VERSION)
manual_review_queue.insert(2, "model_version", MODEL_VERSION)
manual_review_queue.insert(3, "baseline_version", BASELINE_VERSION)
if MANUAL_REVIEW_PATH.is_file():
    existing_manual_review = pd.read_parquet(MANUAL_REVIEW_PATH)
    expected_identity = {
        "dataset_version": DATASET_VERSION,
        "split_version": SPLIT_VERSION,
        "model_version": MODEL_VERSION,
        "baseline_version": BASELINE_VERSION,
    }
    for column, expected in expected_identity.items():
        assert set(existing_manual_review[column].dropna().unique()) == {expected}
    manual_review_queue = preserve_manual_review_labels(
        manual_review_queue,
        existing_manual_review,
    )
normalized_human_fields = manual_review_queue[
    list(MANUAL_LABEL_COLUMNS)
].fillna("").astype(str).apply(lambda column: column.str.strip())
incomplete_review_rows = normalized_human_fields.eq("").any(axis=1)
if incomplete_review_rows.any():
    manual_review_summary = {
        "status": "pending_human_review",
        "row_count": int(len(manual_review_queue)),
        "completed_row_count": int((~incomplete_review_rows).sum()),
        "incomplete_row_count": int(incomplete_review_rows.sum()),
        "selection_scope": {
            "source": (
                "stratified sample of this model's disagreements with "
                "rating-derived weak labels"
            ),
            "is_representative_accuracy_sample": False,
            "historical_labels_transferred": False,
        },
    }
else:
    manual_review_summary = summarize_manual_review(manual_review_queue)

manual_queue_counts = (
    manual_review_queue["suggested_review_group"]
    .value_counts()
    .rename_axis("suggested_review_group")
    .rename("review_count")
    .reset_index()
)
display(manual_queue_counts)
if manual_review_summary["status"] == "completed":
    manual_review_overview = pd.DataFrame(
        [
            {
                "comparison": "DistilBERT vs human",
                **manual_review_summary["model_human_agreement"],
            },
            {
                "comparison": "Rating-derived label vs human",
                **manual_review_summary["rating_label_human_agreement"],
            },
            {
                "comparison": (
                    "DistilBERT vs human when rating marked mismatch"
                ),
                **manual_review_summary[
                    "model_human_agreement_when_rating_marked_mismatch"
                ],
            },
        ]
    )
    display(manual_review_overview)
    display(pd.DataFrame(manual_review_summary["by_suggested_review_group"]))
else:
    display(pd.Series(manual_review_summary, name="value").to_frame())

## 10. Save reproducible results

Only model errors are copied into the detailed Parquet file; correct rows remain available in the original versioned prediction file. This avoids duplicating all review text. The JSON report contains counts and measurements but no review text.

## Сохраняем результаты

В подробный Parquet-файл копируются только ошибки модели; правильные строки уже есть в исходном файле предсказаний с указанной версией. Так мы не дублируем все тексты отзывов. JSON-отчёт содержит числа и измерения, но не тексты отзывов.

In [ ]:
# Save compact versioned files and immediately verify their row counts.
# Сохраняем компактные файлы с версиями и сразу проверяем количество строк.
ERROR_COLUMNS = [
    "review_id",
    "split_name",
    "sentiment_label",
    "predicted_label",
    "rating",
    "seller_attention",
    "seller_attention_policy_version",
    "model_score",
    "score_negative",
    "score_neutral",
    "score_positive",
    "parent_asin",
    "asin",
    "user_id",
    "review_timestamp",
    "product_title",
    "category_path_text",
    "token_count",
    "was_truncated",
    "error_pair",
    "possible_text_rating_disagreement",
    "contains_mixed_opinion_clue",
    "mentions_delivery_or_seller",
    "no_clear_dictionary_sentiment",
    "very_short_review",
    "strong_emotion_clue",
    "review_text",
]
def write_parquet_atomically(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    frame.to_parquet(temporary_path, index=False)
    os.replace(temporary_path, path)


def write_json_atomically(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(json_ready(payload), indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary_path, path)


ERROR_DIRECTORY.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
write_parquet_atomically(model_errors[ERROR_COLUMNS], ERROR_ROWS_PATH)
write_parquet_atomically(manual_review_queue, MANUAL_REVIEW_PATH)
temporary_csv_path = MANUAL_REVIEW_CSV_PATH.with_suffix(
    MANUAL_REVIEW_CSV_PATH.suffix + ".tmp"
)
escape_dataframe_for_spreadsheet(manual_review_queue).to_csv(
    temporary_csv_path, index=False
)
os.replace(temporary_csv_path, MANUAL_REVIEW_CSV_PATH)

error_report = {
    "report_version": "sentiment_error_analysis_v3",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_version": DATASET_VERSION,
    "split_version": SPLIT_VERSION,
    "model_version": MODEL_VERSION,
    "baseline_version": BASELINE_VERSION,
    "evaluation_splits": list(EVALUATION_SPLITS),
    "evaluation_review_count": int(len(analysis_rows)),
    "model_error_count": int(analysis_rows["is_error"].sum()),
    "seller_attention_summary": seller_attention_summary.to_dict(
        orient="records"
    ),
    "split_error_rates": split_error_rates.to_dict(orient="records"),
    "class_error_rates": class_error_rates.to_dict(orient="records"),
    "class_metrics": class_metrics.to_dict(orient="records"),
    "error_pairs": error_pairs.to_dict(orient="records"),
    "calibration": calibration_summary,
    "calibration_by_split": calibration_by_split,
    "calibration_by_model_answer": calibration_by_answer,
    "calibration_bins": calibration_table.assign(
        score_range=calibration_table["score_range"].astype(str)
    ).to_dict(orient="records"),
    "truncation_error_rates": truncation_error_rates.to_dict(orient="records"),
    "possible_cause_indicators": indicator_summary.to_dict(orient="records"),
    "saved_model_comparison_coverage": comparison_coverage.to_dict(
        orient="records"
    ),
    "saved_model_comparison": model_comparison_summary.to_dict(orient="records"),
    "manual_review": {
        **manual_review_summary,
        "path": str(MANUAL_REVIEW_PATH.relative_to(PROJECT_ROOT)),
        "csv_path": str(
            MANUAL_REVIEW_CSV_PATH.relative_to(PROJECT_ROOT)
        ),
        "summary_path": str(MANUAL_SUMMARY_PATH.relative_to(PROJECT_ROOT)),
    },
    "model_decision": {
        "status": (
            "candidate_review_completed"
            if manual_review_summary["status"] == "completed"
            else "candidate_pending_human_review"
        ),
        "decision": (
            "Do not inherit acceptance from the historical model. "
            "Evaluate this corrected-lineage version on its own evidence."
        ),
        "evidence": [
            "All automated analysis uses the corrected UTC split lineage.",
            "DistilBERT and TF-IDF predictions are joined on exact review IDs.",
            (
                "The new diagnostic queue is complete."
                if manual_review_summary["status"] == "completed"
                else "The new diagnostic queue still requires human labels."
            ),
        ],
        "limitations": [
            "The manual queue is a diagnostic disagreement sample, not an unbiased accuracy benchmark.",
            "Neutral model scores must not be presented as calibrated human-confidence values.",
            "Seller attention remains a separate deterministic rating signal.",
        ],
    },
    "error_rows_path": str(ERROR_ROWS_PATH.relative_to(PROJECT_ROOT)),
}
write_json_atomically(manual_review_summary, MANUAL_SUMMARY_PATH)
write_json_atomically(error_report, REPORT_PATH)

assert len(pd.read_parquet(ERROR_ROWS_PATH)) == len(model_errors)
assert len(pd.read_parquet(MANUAL_REVIEW_PATH)) == len(manual_review_queue)
saved_report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
assert saved_report["model_version"] == MODEL_VERSION
assert saved_report["model_error_count"] == len(model_errors)
assert saved_report["manual_review"]["status"] == (
    manual_review_summary["status"]
)
assert saved_report["model_decision"]["status"] != (
    "accepted_v1_with_limitations"
)

display(
    pd.Series(
        {
            "evaluated_reviews": len(analysis_rows),
            "saved_error_rows": len(model_errors),
            "manual_review_rows": len(manual_review_queue),
            "manual_review_status": manual_review_summary["status"],
            "error_rows_path": str(ERROR_ROWS_PATH.relative_to(PROJECT_ROOT)),
            "manual_review_path": str(MANUAL_REVIEW_PATH.relative_to(PROJECT_ROOT)),
            "manual_review_csv_path": str(
                MANUAL_REVIEW_CSV_PATH.relative_to(PROJECT_ROOT)
            ),
            "manual_summary_path": str(
                MANUAL_SUMMARY_PATH.relative_to(PROJECT_ROOT)
            ),
            "report_path": str(REPORT_PATH.relative_to(PROJECT_ROOT)),
        },
        name="value",
    ).to_frame()
)

## Conclusion

The notebook analyzes 209,021 corrected-lineage final-test reviews without rerunning either model. Exact DistilBERT/TF-IDF comparisons, calibration measurements, class errors, product slices, and cause indicators are stored in the new versioned report; historical point estimates are not reused.

A fresh deterministic diagnostic queue is created for this model version. Until its human fields are complete, the summary remains `pending_human_review`: no historical agreement count or acceptance decision is transferred. Even after completion, this disagreement-focused sample will not be an unbiased corpus-wide accuracy estimate.

The corrected DistilBERT remains a candidate pending its own human review. Its score is a model output rather than guaranteed human confidence, especially for Neutral. Rating-derived labels remain weak labels rather than human ground truth. Seller attention remains independent: 1–3 stars are `needs_attention`, while 4–5 stars are `satisfied`.

## Вывод

Ноутбук анализирует 209 021 final-test отзыв исправленного lineage без повторного запуска моделей. Точные сравнения DistilBERT/TF-IDF, calibration, ошибки классов, product-срезы и индикаторы причин сохраняются в новом версионированном отчёте; исторические оценки не переиспользуются.

Для этой версии модели создаётся новая детерминированная диагностическая очередь. Пока её human-поля не заполнены, summary остаётся `pending_human_review`: исторические agreement counts и решение о принятии не переносятся. Даже после завершения выборка расхождений не будет несмещённой оценкой общей accuracy.

Исправленная DistilBERT остаётся кандидатом до собственной human review. Её score — выход модели, а не гарантированная человеческая уверенность, особенно для Neutral. Метки по рейтингу остаются слабыми метками, а не человеческой истиной. Сигнал продавцу независим: 1–3 звезды означают `needs_attention`, а 4–5 звёзд — `satisfied`.

In [ ]:
display(
    manual_review_queue[
        [
            "review_id",
            "sentiment_label",
            "predicted_label",
            "human_text_sentiment",
            "human_rating_matches_text",
            "human_error_cause",
        ]
    ]
)

## Conclusion: versioned manual-review queue

The preceding display shows the current fields of the new queue. The versioned Parquet file is the row-level source, the escaped CSV is the human-facing worksheet, and the JSON summary reports completion status without review text.

## Вывод: версионированная очередь ручной проверки

Предыдущая таблица показывает текущее состояние новой очереди. Версионированный Parquet — построчный источник, защищённый CSV — worksheet для человека, а JSON-summary хранит статус завершения без текстов отзывов.